In [ ]:
import pandas
from matplotlib import pyplot
def fixed_diff(data):
    fixed = (data[data["id"] == "fixed"][["seed", "round", "accuracy", "loss"]]
        .rename(columns={"accuracy": "fixed_accuracy", "loss": "fixed_loss",}))
    data_diff = data.merge(fixed, on=["seed", "round"])
    data_diff["accuracy"] = data_diff["accuracy"] - data_diff["fixed_accuracy"]
    data_diff["loss"] = data_diff["loss"] - data_diff["fixed_loss"]
    data_diff = data_diff.drop(columns=["fixed_accuracy", "fixed_loss"])
    return data_diff
def flwr_accuracy_error(file, start=0, key="accuracy"):
    data = pandas.read_csv(file + ".csv")
    data = data[data["round"] >= start]
    data = fixed_diff(data)
    summary = (data.groupby(["id", "round"], sort=False)[key].agg(mean="mean", std="std").reset_index())
    pyplot.figure(figsize=(11, 6))
    for id_val, group in summary.groupby("id", sort=False):
        group = group.sort_values("round")
        if id_val == "fixed":
            pyplot.plot(group["round"], group["mean"], label=str(id_val))
        else:
            pyplot.errorbar(group["round"], group["mean"], yerr=group["std"], linewidth=2, capsize=4, alpha=0.7, label=str(id_val))
    pyplot.xlabel("round")
    pyplot.ylabel(key)
    pyplot.grid(True, alpha=0.3)
    pyplot.legend()
    pyplot.tight_layout()
    pyplot.show()

In [ ]:
import pandas
from matplotlib import pyplot
def flwr_accuracy(file, start=0, key="accuracy"):
    data = pandas.read_csv(file + ".csv")
    data = data[data["round"] >= start]
    summary = (data.groupby(["id", "round"], sort=False)[key].agg(mean="mean", std="std").reset_index())
    figure = pyplot.figure(figsize=(11, 6))
    for id_val, group in summary.groupby("id", sort=False):
        group = group.sort_values("round")
        line, = pyplot.plot(group["round"], group["mean"], label=str(id_val))
        pyplot.fill_between(
        group["round"],
        group["mean"] - group["std"],
        group["mean"] + group["std"],
        color=line.get_color(),
        alpha=0.2
    )

    pyplot.xlabel("round")
    pyplot.ylabel(key)
    pyplot.grid(True, alpha=0.3)
    pyplot.legend()
    pyplot.tight_layout()
    pyplot.show()
    # figure.savefig(file + "_" + key + "_diagram.pdf", bbox_inches="tight")

In [ ]:
def accuracy_summary(file):
    data = pandas.read_csv(file + ".csv")
    data = data[data["round"] > 190]
    fixed = (
        data[data["id"] == "fixed"]
        [["seed", "round", "accuracy"]]
        .rename(columns={"accuracy": "fixed"})
    )
    data = data.merge(fixed, on=["seed", "round"], how="left")
    data["fixed"] = data["accuracy"] - data["fixed"]
    summary = (data.groupby("id", sort=False)["accuracy"].agg(mean="mean", deviation="std").reset_index())
    summary_diff = (data.groupby("id", sort=False)["fixed"].agg(diff="mean", dev_diff="std").reset_index())
    summary = summary.merge(summary_diff, on=["id"], how="left")
    summary = summary.set_index("id")
    summary.index.name = None
    summary = summary.style.format("{:.4f}")
    return summary

In [ ]:
flwr_accuracy("flwr_all", 160)

In [ ]:
flwr_accuracy_error("flwr_all", 160)

In [ ]:
accuracy_summary("flwr_all")

In [ ]:
accuracy_summary("flwr_sub")

In [ ]:
flwr_accuracy("flwr_sub", 160)

In [ ]:
flwr_accuracy("flwr_all", 160, "loss")